# C03 — Cylindrical meta-atom

## Mathematical and physical model

The structure is nonmagnetic and periodic in the transverse plane. In every constant-$z$ slice the electric field satisfies
$$
\nabla\times\nabla\times\mathbf E-k_0^2\epsilon_r(\mathbf r)\mathbf E=0,\qquad
\mathbf E_t(\mathbf r)=\sum_{m,n}\mathbf e_{mn}(z)e^{i(\mathbf k_{\parallel,0}+\mathbf G_{mn})\cdot (x,y)},\qquad
\mathbf G_{mn}=2\pi(m/P_x,n/P_y).
$$
Match tangential $\mathbf E$ and $\mathbf H$ across the layer interfaces. In each exterior medium $j$, the outgoing order is propagating when
$k_{z,j,mn}^2=k_0^2\epsilon_j-|\mathbf k_{\parallel,0}+\mathbf G_{mn}|^2>0$.
The comparison uses the complex amplitudes normalized so $|a_{mn}^{q\leftarrow p}|^2$ is the incident-power fraction in an outgoing polarization channel. A one-dimensional grating retains $n=0$ even when the incident $k_y$ is nonzero.

Use $\epsilon_r=\epsilon_{\rm pillar}$ inside $x^2+y^2<r^2$ within the patterned layer and the background permittivity outside. The radius sweep follows $T_{00}(r)=|t_{00}(r)|^2$ and $\arg t_{00}(r)$ at fixed wavelength. Fourfold symmetry constrains the normal-incidence polarization response.

## Physical scenario

A square array of cylindrical dielectric pillars is illuminated at fixed wavelength while the pillar radius varies. Changing radius changes the modal response and therefore the transmission amplitude and phase. The symmetric nominal cell also makes the two linear polarizations a useful consistency check.

## Possible applications

The radius sweep is the basis of a unit-cell library for metalenses and other phase-gradient metasurfaces. Design would select geometries that supply the required phase while maintaining useful transmission.

## Benchmark objective

Compare transmitted phase and efficiency as the radius of a dielectric pillar changes.

## Reproduce

Use the project environment and select **Run All**. The notebook and CLI use [case.yaml](case.yaml). The native [Lumerical model](model.fsp) and [reconstruction script](model.lsf) correspond to quick-sweep point 0.

[Published comparisons and figures](../../../BENCHMARK_REPORT.md#c03) · [Periodic diagnosis](../../../PERIODIC_DIAGNOSIS.md#c03)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("C03")[0]
samples, settings, metadata = case_setup(case_path)
results = {}

print(f"{len(samples)} points, Fourier truncation index {settings.order}")
figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe, solve

print(probe())
results["meent"] = solve_sweep(solve, samples, settings)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe, solve

print(probe())
results["torcwa"] = solve_sweep(solve, samples, settings)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe, solve

print(probe())
results["grcwa"] = solve_sweep(solve, samples, settings)

## Lumerical RCWA

This cell uses the installed, licensed FDTD application through its Python API. It reuses one session
across the sweep and evaluates the same model generator used by the reconstruction script.

In [ ]:
from rcwa_cv.solvers.lumerical import session, solve

with session() as fdtd:
    print(f"Lumerical build {fdtd.version()}")
    results["lumerical"] = solve_sweep(solve, samples, settings)

## Compare the solvers

The spectra show total reflection, transmission and the co-polarized transmission phase.
The order plot uses a point shared by the solvers. The heatmaps show maximum discrepancies
over the tested points. Periodic quick sweeps provide preliminary comparisons.

In [ ]:
show_versions(results)
show_results(metadata, results)